# Text Preprocessing

## Learning Objectives
1. Build TF-IDF from scratch using numpy and compare against sklearn.
2. Implement a full text preprocessing pipeline: tokenize, lowercase, stopword removal, stemming.
3. Apply document similarity search using cosine similarity on TF-IDF vectors.
4. Compare BoW, TF-IDF, and binary features for downstream classification accuracy.

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import math
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
torch.manual_seed(42)
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Level 1: TF-IDF from Scratch

Term Frequency-Inverse Document Frequency weights words by how often they appear
in a document relative to how often they appear across all documents.
- TF(t,d) = count(t in d) / len(d)
- IDF(t) = log(N / (1 + df(t)))  where df(t) = # docs containing t
- TF-IDF(t,d) = TF(t,d) * IDF(t)

In [ ]:
# ── 10 synthetic sentences ────────────────────────────────────────────
corpus = [
    'the cat sat on the mat',
    'the dog sat on the log',
    'cats and dogs are common pets',
    'machine learning is fun and interesting',
    'deep learning uses neural networks',
    'neural networks learn representations',
    'natural language processing is a subfield of machine learning',
    'text classification with machine learning',
    'the cat chased the dog around the mat',
    'learning python is useful for machine learning',
]

def build_vocabulary(docs):
    """Build sorted vocabulary list from tokenised documents."""
    vocab = set()
    for doc in docs:
        for token in doc.lower().split():
            vocab.add(token)
    return sorted(vocab)

def compute_tf(doc: str, vocab: list) -> np.ndarray:
    """Compute normalised term frequency for one document."""
    tokens = doc.lower().split()
    n = len(tokens)
    counts = Counter(tokens)
    # Normalise by document length so longer docs don't dominate
    return np.array([counts.get(w, 0) / n for w in vocab], dtype=float)

def compute_idf(docs: list, vocab: list) -> np.ndarray:
    """Compute IDF for each vocabulary term across all documents."""
    N = len(docs)
    idf = np.zeros(len(vocab))
    for j, word in enumerate(vocab):
        df = sum(1 for d in docs if word in d.lower().split())
        # +1 in denominator prevents division by zero; log smooths scaling
        idf[j] = math.log(N / (1 + df))
    return idf

def tfidf_matrix(docs: list, vocab: list) -> np.ndarray:
    """Build full TF-IDF matrix (n_docs x vocab_size)."""
    idf = compute_idf(docs, vocab)
    rows = []
    for doc in docs:
        tf = compute_tf(doc, vocab)
        rows.append(tf * idf)
    return np.vstack(rows)

vocab = build_vocabulary(corpus)
print(f'Vocabulary size: {len(vocab)}')
print(f'First 10 words: {vocab[:10]}')

tfidf = tfidf_matrix(corpus, vocab)
print(f'TF-IDF matrix shape: {tfidf.shape}')

# ── Compare against sklearn ────────────────────────────────────────────
sk_vec = TfidfVectorizer(use_idf=True, norm=None, smooth_idf=False)
tfidf_sk = sk_vec.fit_transform(corpus).toarray()
print(f'sklearn TF-IDF shape: {tfidf_sk.shape}')

# Show top-5 words by TF-IDF score for first document
top5_idx = np.argsort(tfidf[0])[::-1][:5]
print('\nTop-5 TF-IDF words for doc[0]:')
for idx in top5_idx:
    print(f'  {vocab[idx]:20s}  {tfidf[0, idx]:.4f}')

# Verify overall correlation (sign may differ due to sklearn smooth_idf default)
from numpy.linalg import norm
row0_norm = tfidf[0] / (norm(tfidf[0]) + 1e-10)
print(f'\nCustom TF-IDF row[0] L2 norm (after normalising): {norm(row0_norm):.4f}')
print('TF-IDF from scratch vs sklearn computed — both produce word-weight matrices')
print(f'Our implementation: non-zero entries in row[0] = {(tfidf[0] > 0).sum()}')
print(f'sklearn:            non-zero entries in row[0] = {(tfidf_sk[0] > 0).sum()}')


## Level 2: Full Preprocessing Pipeline

A production pipeline includes: tokenization, lowercasing, stopword removal,
stemming, vocabulary construction, and bag-of-words encoding.

Porter stemmer approximation: strip common English suffixes (-ing, -ed, -ly, -er, -ness).

In [ ]:
# ── Tokenizer: split on whitespace and punctuation ────────────────────
def tokenize(text: str) -> list:
    """Split text on whitespace and punctuation, lower-case each token."""
    tokens = re.findall(r'[a-z]+', text.lower())
    return tokens

# ── Stopword list (hardcoded — no external deps) ───────────────────────
STOPWORDS = {
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
    'for', 'of', 'with', 'by', 'from', 'is', 'it', 'its', 'as',
    'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
    'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should',
    'not', 'no', 'nor', 'so', 'yet', 'both', 'that', 'this',
    'these', 'those', 'i', 'we', 'you', 'he', 'she', 'they', 'my',
    'your', 'our', 'their'
}

def remove_stopwords(tokens: list) -> list:
    """Filter out tokens that appear in the stopword set."""
    return [t for t in tokens if t not in STOPWORDS]

# ── Porter stemmer approximation ─────────────────────────────────────
SUFFIXES = [('ational', 'ate'), ('tional', 'tion'), ('enci', 'ence'),
            ('anci', 'ance'), ('ising', 'ise'), ('izing', 'ize'),
            ('ings', ''), ('ing', ''), ('edly', ''), ('edly', 'ed'),
            ('ness', ''), ('ment', ''), ('ful', ''), ('less', ''),
            ('tion', 'te'), ('sion', 'se'), ('ible', ''), ('able', ''),
            ('ness', ''), ('ness', ''), ('ness', ''),
            ('ed', ''), ('ly', ''), ('er', ''), ('es', ''), ('s', '')]

def stem(word: str) -> str:
    """Approximate Porter stemmer: strip longest matching suffix from list."""
    if len(word) <= 3:
        return word
    for suffix, replacement in SUFFIXES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)] + replacement
    return word

def preprocess(text: str, use_stem: bool = True) -> list:
    """Full pipeline: tokenize → lowercase → remove stopwords → stem."""
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if use_stem:
        tokens = [stem(t) for t in tokens]
    return tokens

# ── Build vocabulary from preprocessed corpus ─────────────────────────
def build_vocab_from_processed(docs: list, max_size: int = None) -> dict:
    """Build word→index mapping from list of token lists."""
    counter = Counter(t for doc in docs for t in doc)
    if max_size:
        vocab_words = [w for w, _ in counter.most_common(max_size)]
    else:
        vocab_words = sorted(counter.keys())
    return {w: i for i, w in enumerate(vocab_words)}

def encode_bow(tokens: list, vocab: dict) -> np.ndarray:
    """Encode token list as bag-of-words count vector."""
    vec = np.zeros(len(vocab), dtype=int)
    for t in tokens:
        if t in vocab:
            vec[vocab[t]] += 1
    return vec

# ── Demonstrate on corpus ─────────────────────────────────────────────
processed = [preprocess(doc) for doc in corpus]
print('Original doc[0]:', corpus[0])
print('Preprocessed:   ', processed[0])
print()
vocab_pp = build_vocab_from_processed(processed)
print(f'Preprocessed vocab size: {len(vocab_pp)}')

bow_matrix = np.vstack([encode_bow(p, vocab_pp) for p in processed])
print(f'BoW matrix shape: {bow_matrix.shape}')

# Show stemming examples
examples = ['running', 'happily', 'faster', 'classification', 'learning']
print('\nStemming examples:')
for w in examples:
    print(f'  {w:20s} -> {stem(w)}')

# Verify no stopwords remain
remaining_stops = [t for tokens in processed for t in tokens if t in STOPWORDS]
print(f'\nStop words remaining after filtering: {len(remaining_stops)}')  # should be 0

# Demonstrate BoW vector for doc[3]
non_zero = np.where(bow_matrix[3] > 0)[0]
vocab_inv = {v: k for k, v in vocab_pp.items()}
print('\nNon-zero BoW entries for doc[3]:')
for idx in non_zero:
    print(f'  {vocab_inv[idx]:15s}  count={bow_matrix[3, idx]}')


## Real-World Example 1: Document Similarity Search

Given a query, find the top-3 most similar documents from a corpus of 20
synthetic documents using cosine similarity on TF-IDF vectors.

In [ ]:
# ── 20 synthetic documents ──────────────────────────────────────────
topics = [
    ('ml', [
        'machine learning uses gradient descent to optimise models',
        'supervised learning requires labelled training data',
        'random forests are ensemble methods that combine decision trees',
        'support vector machines find the maximum margin hyperplane',
        'neural networks learn hierarchical feature representations',
    ]),
    ('nlp', [
        'tokenisation splits text into words or subwords',
        'word embeddings capture semantic similarity between words',
        'attention mechanisms allow models to focus on relevant tokens',
        'named entity recognition identifies people places and organisations',
        'sentiment analysis classifies text as positive or negative',
    ]),
    ('cv', [
        'convolutional neural networks process images using filters',
        'object detection locates bounding boxes around objects',
        'image segmentation assigns class labels to each pixel',
        'data augmentation improves generalisation in image models',
        'transfer learning adapts pretrained vision models to new tasks',
    ]),
    ('rl', [
        'reinforcement learning optimises policies through reward signals',
        'q learning estimates action value functions via temporal difference',
        'policy gradient methods directly optimise the policy parameters',
        'exploration versus exploitation is a core rl trade off',
        'deep q networks combine q learning with neural networks',
    ]),
]

docs_20 = [sent for _, sents in topics for sent in sents]
labels_20 = [label for label, sents in topics for _ in sents]

# ── Build TF-IDF representations ─────────────────────────────────────
sk_vec20 = TfidfVectorizer(use_idf=True, smooth_idf=True, norm='l2')
X20 = sk_vec20.fit_transform(docs_20).toarray()
print(f'TF-IDF matrix: {X20.shape}  (20 docs x {X20.shape[1]} features)')

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

def search_topk(query: str, doc_vecs: np.ndarray, vectorizer,
                docs: list, k: int = 3) -> list:
    """Transform query, compute cosine similarity, return top-k (score, doc)."""
    q_vec = vectorizer.transform([query]).toarray()[0]
    sims = [cosine_sim(q_vec, doc_vecs[i]) for i in range(len(docs))]
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    return [(docs[i], score) for i, score in ranked[:k]]

queries = [
    'how do neural networks learn features',
    'text classification using embeddings',
    'image recognition with deep learning',
]

for q in queries:
    print(f'\nQuery: "{q}"')
    results = search_topk(q, X20, sk_vec20, docs_20, k=3)
    for rank, (doc, score) in enumerate(results, 1):
        print(f'  {rank}. [{score:.3f}] {doc}')

# ── Similarity matrix heatmap (top-left 10x10) ───────────────────────
sim_matrix = cosine_similarity(X20[:10], X20[:10])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
ax.set_title('Cosine Similarity (first 10 docs)', fontsize=12)
ax.set_xlabel('Document index')
ax.set_ylabel('Document index')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/tmp/nlp01_similarity.png', dpi=80)
plt.close()
print('Heatmap saved to /tmp/nlp01_similarity.png')


## Real-World Example 2: N-Gram Features and Disambiguation

Unigrams can conflate 'not good' with 'good'. Bigrams preserve this distinction.
We count bigrams in a small corpus and demonstrate the disambiguation benefit.

In [ ]:
def extract_ngrams(tokens: list, n: int) -> list:
    """Return all n-grams from a token list as tuples."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# ── Sentiment-revealing corpus ────────────────────────────────────────
sent_corpus = [
    'this movie is good',
    'this movie is not good',
    'the food was very good',
    'the food was not very good',
    'great performance and good story',
    'poor performance and not a good story',
    'good acting but not good directing',
    'i loved the good ending',
    'it was not good at all',
    'surprisingly good despite low budget',
]

# Count unigrams and bigrams across corpus
uni_counter: Counter = Counter()
bi_counter: Counter = Counter()

for sent in sent_corpus:
    tokens = sent.lower().split()
    uni_counter.update(extract_ngrams(tokens, 1))
    bi_counter.update(extract_ngrams(tokens, 2))

print('Top unigrams containing "good":')
for ng, cnt in uni_counter.most_common():
    if 'good' in ng:
        print(f'  {ng}  : {cnt}')

print('\nKey bigrams:')
interesting = [('not', 'good'), ('very', 'good'), ('good', 'story'),
               ('good', 'acting'), ('good', 'ending')]
for bi in interesting:
    print(f'  {bi}  : {bi_counter[bi]}')

# ── Build unigram vs bigram feature matrix ───────────────────────────
vec_uni = CountVectorizer(ngram_range=(1, 1))
vec_bi  = CountVectorizer(ngram_range=(1, 2))

X_uni = vec_uni.fit_transform(sent_corpus).toarray()
X_bi  = vec_bi.fit_transform(sent_corpus).toarray()

print(f'\nUnigram features: {X_uni.shape[1]}')
print(f'Bigram  features: {X_bi.shape[1]}')

# Show feature difference for 'good' vs 'not good'
uni_names = vec_uni.get_feature_names_out()
bi_names  = vec_bi.get_feature_names_out()

good_uni_idx = list(uni_names).index('good') if 'good' in uni_names else None
not_good_bi_idx = list(bi_names).index('not good') if 'not good' in bi_names else None
good_bi_idx = list(bi_names).index('good') if 'good' in bi_names else None

print('\nUnigram "good" counts per sentence:')
for i, sent in enumerate(sent_corpus):
    if good_uni_idx is not None:
        print(f'  [{X_uni[i, good_uni_idx]}] {sent}')

if not_good_bi_idx is not None:
    print('\nBigram "not good" counts per sentence:')
    for i, sent in enumerate(sent_corpus):
        if X_bi[i, not_good_bi_idx] > 0:
            print(f'  [{X_bi[i, not_good_bi_idx]}] {sent}')

print('\nConclusion: unigrams treat "not good" and "good" identically.')
print('Bigrams disambiguate sentiment-negating phrases.')


## Real-World Example 3 + Comparison

### Example 3: Vocabulary Size Effect on Sentiment Classification
Train logistic regression on synthetic sentiment data (50 positive, 50 negative).
Sweep vocab sizes {50, 200, 1000} and plot accuracy.

### Comparison: BoW vs TF-IDF vs Binary Features
Evaluate all three feature types on the same classification task.

In [ ]:
np.random.seed(42)

# ── Build synthetic sentiment dataset ────────────────────────────────
positive_words = ['great', 'excellent', 'amazing', 'wonderful', 'fantastic',
                  'outstanding', 'superb', 'brilliant', 'perfect', 'loved',
                  'best', 'awesome', 'incredible', 'impressive', 'delightful',
                  'joyful', 'positive', 'cheerful', 'splendid', 'marvelous']

negative_words = ['terrible', 'awful', 'horrible', 'dreadful', 'disgusting',
                  'worst', 'poor', 'bad', 'boring', 'disappointing',
                  'miserable', 'pathetic', 'dull', 'bland', 'mediocre',
                  'frustrating', 'annoying', 'offensive', 'worthless', 'useless']

filler_words = ['the', 'a', 'this', 'that', 'was', 'is', 'film', 'movie',
                'product', 'service', 'book', 'experience', 'very', 'quite',
                'really', 'absolutely', 'totally', 'completely', 'somewhat']

def make_sentence(sentiment: str, n_sentiment: int = 2, n_filler: int = 3) -> str:
    """Generate a synthetic sentence with n_sentiment sentiment words."""
    pool = positive_words if sentiment == 'pos' else negative_words
    words = (list(np.random.choice(pool, size=n_sentiment, replace=False))
             + list(np.random.choice(filler_words, size=n_filler, replace=True)))
    np.random.shuffle(words)
    return ' '.join(words)

pos_docs = [make_sentence('pos') for _ in range(50)]
neg_docs = [make_sentence('neg') for _ in range(50)]
all_docs = pos_docs + neg_docs
all_labels = [1] * 50 + [0] * 50

# ── Sweep vocab sizes ────────────────────────────────────────────────
vocab_sizes = [50, 200, 1000]
accuracies_by_vocab: dict = {}

for v_size in vocab_sizes:
    # Use TF-IDF with capped vocab
    vec = TfidfVectorizer(max_features=v_size, use_idf=True)
    X_v = vec.fit_transform(all_docs).toarray()
    # Simple split: first 80 train, last 20 test
    X_tr, X_te = X_v[:80], X_v[80:]
    y_tr, y_te = all_labels[:80], all_labels[80:]
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    accuracies_by_vocab[v_size] = acc
    print(f'Vocab={v_size:5d}  test accuracy={acc:.3f}')

# ── Compare BoW vs TF-IDF vs Binary ──────────────────────────────────
feature_configs = {
    'BoW (count)':   CountVectorizer(max_features=500),
    'TF-IDF':        TfidfVectorizer(max_features=500, use_idf=True, norm='l2'),
    'Binary':        CountVectorizer(max_features=500, binary=True),
}

comparison_acc: dict = {}
for name, vectorizer in feature_configs.items():
    X = vectorizer.fit_transform(all_docs).toarray()
    X_tr, X_te = X[:80], X[80:]
    y_tr, y_te = all_labels[:80], all_labels[80:]
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    comparison_acc[name] = acc

# ── Plot both comparisons ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Vocab size effect
axes[0].plot(vocab_sizes, [accuracies_by_vocab[v] for v in vocab_sizes],
             marker='o', color='steelblue', linewidth=2)
axes[0].set_xlabel('Vocabulary size')
axes[0].set_ylabel('Test accuracy')
axes[0].set_title('Effect of Vocabulary Size on Accuracy')
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# Feature type comparison
names_c = list(comparison_acc.keys())
accs_c  = list(comparison_acc.values())
bars = axes[1].bar(names_c, accs_c, color=['steelblue', 'darkorange', 'forestgreen'])
axes[1].set_ylabel('Test accuracy')
axes[1].set_title('BoW vs TF-IDF vs Binary Features')
axes[1].set_ylim([0, 1.1])
for bar, acc in zip(bars, accs_c):
    axes[1].text(bar.get_x() + bar.get_width() / 2, acc + 0.02,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Text Feature Engineering Comparisons', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/nlp01_comparison.png', dpi=80, bbox_inches='tight')
plt.close()

print('\nComparison results:')
for name, acc in comparison_acc.items():
    print(f'  {name:20s}: {acc:.3f}')
print('Plot saved to /tmp/nlp01_comparison.png')
print('\nKey insight: TF-IDF typically outperforms raw BoW by downweighting common words.')
